In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.project_21a;

In [0]:
%sql
USE SCHEMA project_21a;

In [0]:
%sql
SELECT CURRENT_CATALOG(),current_schema();

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS project_21a_data;

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_inventory.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_sales.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DIM_STORE
(
    STORE_KEY BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY,
    STORE_ID STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DIM_PRODUCT
(
    PRODUCT_KEY BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) PRIMARY KEY,
    PRODUCT_ID STRING
);

In [0]:
%sql
INSERT INTO DIM_STORE(STORE_ID)
SELECT STORE_ID FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_inventory.json',
    FORMAT => 'json')
WHERE STORE_ID IS NOT NULL
UNION 
SELECT DISTINCT STORE_ID FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_sales.json',
    FORMAT => 'json'
)
WHERE STORE_ID IS NOT NULL;

In [0]:
%sql
INSERT INTO DIM_PRODUCT(PRODUCT_ID)
SELECT PRODUCT_ID FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_inventory.json',
    FORMAT => 'json')
WHERE PRODUCT_ID IS NOT NULL
UNION 
SELECT PRODUCT_ID FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_sales.json',
    FORMAT => 'json'
)
WHERE PRODUCT_ID IS NOT NULL;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS RAW_SALES
(
    SALE_ID STRING,
    STORE_ID STRING,
    PRODUCT_ID STRING,
    AMOUNT DECIMAL(10,2),
    DATE_KEY BIGINT
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS RAW_INVENTORY
(
    SNAPSHOT_ID STRING,
    STORE_ID STRING,
    PRODUCT_ID STRING,
    ON_HAND_QTY INT,
    DATE_KEY BIGINT
);

In [0]:
%sql
INSERT INTO RAW_SALES(SALE_ID,STORE_ID,PRODUCT_ID,AMOUNT,DATE_KEY)
SELECT SALE_ID,
       STORE_ID,
       PRODUCT_ID,
       AMOUNT,
       DATE_KEY
FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_sales.json',
    FORMAT => 'json'
);

In [0]:
%sql
INSERT INTO RAW_INVENTORY(SNAPSHOT_ID,STORE_ID,PRODUCT_ID,ON_HAND_QTY,DATE_KEY)
SELECT SNAPSHOT_ID,
       STORE_ID,
       PRODUCT_ID,
       ON_HAND_QTY,
       DATE_KEY 
FROM read_files(
    '/Volumes/workspace/project_21a/project_21a_data/raw_inventory.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT  'POS_SALES' AS DATASET,
        COUNT(*) AS LOADED_ROWS
FROM RAW_SALES
UNION ALL
SELECT  'INVENTORY' AS DATASET,
        COUNT(*) AS LOADED_ROWS
FROM RAW_INVENTORY;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS FACT_SALES
(
    SALE_ID STRING PRIMARY KEY,
    STORE_KEY BIGINT REFERENCES DIM_STORE(STORE_KEY),
    PRODUCT_KEY BIGINT REFERENCES DIM_PRODUCT(PRODUCT_KEY),
    AMOUNT DECIMAL(10,2),
    DATE_KEY BIGINT
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS FACT_INVENTORY
(
    SNAPSHOT_ID STRING PRIMARY KEY,
    STORE_KEY BIGINT REFERENCES DIM_STORE(STORE_KEY),
    PRODUCT_KEY BIGINT REFERENCES DIM_PRODUCT(PRODUCT_KEY),
    ON_HAND_QTY INT,
    DATE_KEY BIGINT
);

In [0]:
%sql
INSERT INTO FACT_SALES(SALE_ID,STORE_KEY,PRODUCT_KEY,AMOUNT,DATE_KEY)
SELECT RS.SALE_ID,
       DS.STORE_KEY,
       DP.PRODUCT_KEY,
       RS.AMOUNT,
       RS.DATE_KEY
FROM RAW_SALES RS
JOIN DIM_STORE DS
ON RS.STORE_ID=DS.STORE_ID
JOIN DIM_PRODUCT DP
ON RS.PRODUCT_ID = DP.PRODUCT_ID

In [0]:
%sql
INSERT INTO FACT_INVENTORY(SNAPSHOT_ID,STORE_KEY,PRODUCT_KEY,ON_HAND_QTY,DATE_KEY)
SELECT RI.SNAPSHOT_ID,
       DS.STORE_KEY,
       DP.PRODUCT_KEY,
       RI.ON_HAND_QTY,
       RI.DATE_KEY
FROM RAW_INVENTORY RI
JOIN DIM_STORE DS
ON RI.STORE_ID=DS.STORE_ID
JOIN DIM_PRODUCT DP
ON RI.PRODUCT_ID = DP.PRODUCT_ID

In [0]:
%sql
SELECT 'Sales Analysis' AS BUSINESS_PROCESS,
CASE WHEN COUNT(FS.SALE_ID) = COUNT(DS.STORE_KEY)
THEN 'YES' ELSE 'NO' END AS CONFORMED_STORE,
CASE WHEN COUNT(FS.SALE_ID) = COUNT(DP.PRODUCT_KEY)
THEN 'YES' ELSE 'NO' END AS CONFORMED_PRODUCT
FROM FACT_SALES FS
LEFT JOIN DIM_STORE DS
ON FS.STORE_KEY = DS.STORE_KEY
LEFT JOIN DIM_PRODUCT DP
ON FS.PRODUCT_KEY = DP.PRODUCT_KEY

UNION ALL

SELECT 'Inventory Snapshot' AS BUSINESS_PROCESS,
CASE WHEN COUNT(FI.SNAPSHOT_ID) = COUNT(DS.STORE_KEY)
THEN 'YES' ELSE 'NO' END AS CONFORMED_STORE,
CASE WHEN COUNT(FI.SNAPSHOT_ID) = COUNT(DP.PRODUCT_KEY)
THEN 'YES' ELSE 'NO' END AS CONFORMED_PRODUCT
FROM FACT_INVENTORY FI
LEFT JOIN DIM_STORE DS
ON FI.STORE_KEY = DS.STORE_KEY
LEFT JOIN DIM_PRODUCT DP
ON FI.PRODUCT_KEY = DP.PRODUCT_KEY;

In [0]:
%sql
CREATE VIEW VW_SALES_DATA_MART AS
SELECT  FS.SALE_ID,
        DS.STORE_ID,
        DP.PRODUCT_ID,
        FS.AMOUNT
FROM FACT_SALES FS
JOIN DIM_STORE DS
ON FS.STORE_KEY = DS.STORE_KEY
JOIN DIM_PRODUCT DP
ON FS.PRODUCT_KEY = DP.PRODUCT_KEY;


In [0]:
%sql
SELECT * FROM VW_SALES_DATA_MART;

In [0]:
%sql
CREATE VIEW VW_INVENTORY_DATA_MART AS
SELECT  FI.SNAPSHOT_ID,
        DS.STORE_ID,
        DP.PRODUCT_ID,
        FI.ON_HAND_QTY
FROM FACT_INVENTORY FI
JOIN DIM_STORE DS
ON FI.STORE_KEY = DS.STORE_KEY
JOIN DIM_PRODUCT DP
ON FI.PRODUCT_KEY = DP.PRODUCT_KEY;

In [0]:
%sql
SELECT * FROM VW_INVENTORY_DATA_MART;

In [0]:
%sql
CREATE TEMPORARY TABLE INCOMING 
(
    SALE_ID STRING,
    AMOUNT DECIMAL(10,2)
);

INSERT INTO INCOMING VALUES
('SL-301',220.00);

In [0]:
%sql
MERGE INTO FACT_SALES T
USING INCOMING S
ON T.SALE_ID=S.SALE_ID

WHEN MATCHED THEN
    UPDATE 
        SET T.AMOUNT=S.AMOUNT
WHEN NOT MATCHED THEN
    INSERT 
        (SALE_ID,AMOUNT)
    VALUES
        (S.SALE_ID,S.AMOUNT);

In [0]:
%sql

SELECT  'FACT_DIM_FK_MATCH' AS INTEGRITY_CHECK,
        COUNT(*) AS FAILED_RECORDS
FROM
(
SELECT FS.SALE_ID
FROM FACT_SALES FS
LEFT JOIN DIM_STORE DS
ON FS.STORE_KEY = DS.STORE_KEY
LEFT JOIN DIM_PRODUCT DP
ON FS.PRODUCT_KEY = DP.PRODUCT_KEY
WHERE DS.STORE_KEY IS NULL
OR DP.PRODUCT_KEY IS NULL

) MISMATCHES;


In [0]:
%sql

OPTIMIZE DIM_STORE ZORDER BY (STORE_KEY);

In [0]:
%sql

OPTIMIZE DIM_PRODUCT ZORDER BY (PRODUCT_KEY);

In [0]:
%sql

SELECT  'STORE_PRODUCT_KEYS' AS CLUSTERING_TARGET,
        'COMPLETED' AS STATUS;